# Example 01.06 — run batch scoring

Scores all 100,000 rows and stores an immutable prediction dataset. Re-running the
notebook reuses the successful operation rather than producing duplicate predictions.


In [ ]:
from pathlib import Path
import sys

REPOSITORY_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "ml_app_client").is_dir()),
    None,
)
if REPOSITORY_ROOT is None:
    raise RuntimeError("Start Jupyter inside the ml-app repository")
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from ml_app_client import MLAppClient, ResourceNotFoundError

client = MLAppClient.connect()
print("Connected to ML App")


## Choose resource names

These are normal names passed to `ml_app_client`. Edit them directly and use the
same values in the following notebooks. Dataset and pipeline names are resolved
inside the selected Business Case. Business Case and service names are globally
unique on one ML App installation.


In [ ]:
# These are ordinary platform resource names. Change the two globally unique
# names (Business Case and model service) when sharing one installation.
BUSINESS_CASE_NAME = "[MLAPP EXAMPLE 01 v2] Estates Lifecycle - demo"
TRAINING_DATASET_NAME = "Example01 Estates - Training"
SCORING_DATASET_NAME = "Example01 Estates - Batch Input"
ACTUALS_DATASET_NAME = "Example01 Estates - Actuals"
TRAINING_PIPELINE_NAME = "Example01 03 - AutoML Training"
BATCH_PIPELINE_NAME = "Example01 05 - Batch Scoring"
MONITORING_PIPELINE_NAME = "Example01 07 - Performance Monitoring"
MODEL_NAME = "Example01 Estates Price Model"
OUTPUT_NAME_PREFIX = "Example01 Estates AutoML"
MODEL_SERVICE_NAME = "Example01 10 - Estates Model Service - demo"

TRAINING_RUN_KEY = "Example01-training-v2"
BATCH_RUN_KEY = "Example01-batch-scoring-v2"
MONITORING_RUN_KEY = "Example01-monitoring-v2"

print("Resource names configured for:", BUSINESS_CASE_NAME)


In [ ]:
try:
    run = client.pipeline_run_by_operation_key(business_case_name=BUSINESS_CASE_NAME, pipeline_name=BATCH_PIPELINE_NAME, operation_key=BATCH_RUN_KEY)
    started = False
except ResourceNotFoundError:
    run = client.run_pipeline_by_name(business_case_name=BUSINESS_CASE_NAME, pipeline_name=BATCH_PIPELINE_NAME, runtime_parameters={"client_operation_key": BATCH_RUN_KEY})
    started = True
print(f"{'STARTED' if started else 'REUSED'} run {run.id}; status={run.status}")
finished = client.wait_for_pipeline_run(run, timeout=3600)

prediction_dataset_id = client.prediction_dataset_id(finished)
preview = client.preview_dataset(prediction_dataset_id, limit=5)
print({
    "run_id": finished.id,
    "processed_rows": finished.processed_row_count,
    "prediction_dataset_id": prediction_dataset_id,
    "total_prediction_rows": preview["row_count"],
    "preview_rows": preview["returned_count"],
})
preview["records"]
